주택 가격 데이터를 활용한 회귀 모델 학습 및 예측

In [16]:
import pandas as pd

df = pd.read_csv('train.csv')
null_counts = df.isnull().sum()

print(null_counts[null_counts > 0].sort_values(ascending=False))

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64


In [ ]:
drop_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu']
df = df.drop(columns=drop_cols)

In [ ]:
# LotFrontage 결측치 부분(nan)은 평균으로 대체
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].mean())

In [ ]:
# 이거 위주로 pd.get_dummies할려했지만 그냥 전체에서 실행하기로 함
# df.select_dtypes(include=['object']).columns

C:\Users\EZ\AppData\Local\Temp\ipykernel_2840\549554427.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include=['object']).columns


Index(['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
       'PavedDrive', 'SaleType', 'SaleCondition'],
      dtype='str')

최종 코드

In [2]:
import pandas as pd
import numpy as np
# 데이터 분리용 train_test_split (학습 데이터와 검증용 테스트 데이터 분할)
from sklearn.model_selection import train_test_split
# 모델 학습: 회귀 예측을 위한 의사결정나무(Decision Tree) 모델
from sklearn.tree import DecisionTreeRegressor
# 모델 평가 지표: 회귀 모델의 성능을 측정하기 위한 지표들
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. 데이터 로드 및 결측치 확인
# 주택 가격 데이터셋(train.csv)을 Pandas DataFrame 형태로 불러옵니다.
df = pd.read_csv('train.csv')

# 각 열(Column)별로 결측치(NaN)의 개수를 계산합니다.
null_counts = df.isnull().sum()
# 결측치가 1개 이상인 열들을 내림차순으로 정렬하여 데이터 상태를 확인합니다.
# print(null_counts[null_counts > 0].sort_values(ascending=False))


# 2. 데이터 전처리 (결측치 처리 및 불필요한 열 제거)
# 결측치 비율이 너무 높아 모델 학습에 방해가 되거나 정보가 부족한 고유 특성 열들을 삭제 대상으로 지정합니다.
drop_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu']
df = df.drop(columns=drop_cols)

# 대표적인 수치형 데이터인 'LotFrontage'(도로 접면 면적)의 결측치는 해당 열의 '평균값(Mean)'으로 대체합니다.
# 원본 데이터를 바로 변경하기 위해 fillna()의 결과를 다시 해당 열에 할당합니다.
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].mean())

# 단순 식별자 역할을 하는 'Id' 열은 주택 가격 예측에 무의미하므로 제거합니다.
df = df.drop(columns=['Id'])


# 3. 범주형 데이터 인코딩
# 문자열 형태의 범주형 데이터를 컴퓨터가 인식할 수 있도록 원-핫 인코딩(One-Hot Encoding)을 수행합니다.
# dtype=int 옵션을 추가하여 변환된 파생 열들이 True/False(부울형)가 아닌 1과 0(정수형)으로 표현되도록 합니다.
# dtype=int는 없어도 상관없어보임
df_encoded = pd.get_dummies(df, dtype=int)


# 4. 학습/테스트 데이터 분리 (8:2 비율)
# 예측 대상인 정답지(Target) 'SalePrice'(주택 가격) 열을 y로, 이를 제외한 나머지 문제지(Feature)를 X로 분리합니다.
X = df_encoded.drop(columns=['SalePrice'])
y = df_encoded['SalePrice']

# 전체 데이터를 학습용(Train)과 성능 검증용(Test) 데이터로 8:2 비율(test_size=0.2)로 나눕니다.
# random_state=42를 지정하여 코드를 다시 실행해도 동일하게 데이터가 분할되도록(재현성 보장) 합니다.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 5. 모델 초기화 및 학습
# 의사결정나무 회귀 모델을 생성합니다.
# 과적합(Overfitting)을 방지하고 모델의 복잡도를 제어하기 위해 나무의 최대 깊이(max_depth)를 5로 제한합니다.
tree_reg = DecisionTreeRegressor(max_depth=5, random_state=42)

# 준비된 학습 데이터(X_train, y_train)를 바탕으로 모델을 패턴을 학습(훈련)시킵니다.
tree_reg.fit(X_train, y_train)
print("모델 학습이 성공적으로 완료되었습니다!")


# 6. 모델 예측 및 성능 평가
# 학습된 모델에 테스트 문제지(X_test)를 입력하여 예측 가격인 y_pred를 생성합니다.
y_pred = tree_reg.predict(X_test)

# 실제 정답(y_test)과 모델의 예측값(y_pred)을 비교하여 4가지 회귀 평가 지표를 계산합니다.
# MAE (평균절대오차): 예측값과 실제값의 차이(오차)의 절대값 평균
mae = mean_absolute_error(y_test, y_pred)
# MSE (평균제곱오차): 오차를 제곱한 값들의 평균 (큰 오차에 더 큰 패널티를 부여)
mse = mean_squared_error(y_test, y_pred)
# RMSE (평균제곱근오차): MSE에 루트를 씌워 실제 데이터 단위(달러 등)와 단위를 맞춘 지표
rmse = np.sqrt(mse)
# R2 Score (결정계수): 모델이 데이터의 분산을 얼마나 잘 설명하는지 나타내는 지표 (1에 가까울수록 우수한 모델)
r2 = r2_score(y_test, y_pred)

# 최종 성능 평가 결과를 소수점 4자리까지 포맷팅하여 가독성 있게 출력합니다.
print("=== 모델 평가 결과 ===")
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")

모델 학습이 성공적으로 완료되었습니다!
=== 모델 평가 결과 ===
MAE  : 27511.2828
MSE  : 1551633394.6276
RMSE : 39390.7780
R2 Score: 0.7977
